# 3 - GMS Primitives, Calibration, and Embedding SFT

This notebook runs the **real** GMS store: the geometric primitives, the **calibrated** decision gate, the two-channel embedding fine-tuning (v-space rotation, u-space contradiction) and its effect shown by PCA.

In [ ]:
import torch
from pathlib import Path
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore

def _find_store(name="gms_policy_store_geode"):
    for p in [Path.cwd(), *Path.cwd().parents]:
        c = p / "beyond-prompt-and-pray" / "code" / "data" / name
        if c.exists():
            return c
    raise FileNotFoundError(name + " not found; run scripts/build_geode_rag_store.py")

STORE = _find_store()
store = GMSExpertStore(DocGMSConfig(store_path=str(STORE), ingest_mode="regex"),
                       device=torch.device("cpu"))
assert store.load(), "store failed to load"
print("loaded store:", len(store.adapter.relation_to_idx), "relations,",
      len(store.adapter.entity_to_idx), "entities")

## The primitives (real store)
`score_triple` (plausibility as distance), `query_triples` (asserted edges) and `tension_energy` (contradiction).

In [ ]:
d35 = store.score_triple('overdraft', 'has_fee_amount', '35.0')
d50 = store.score_triple('overdraft', 'has_fee_amount', '50.0')
print(f'score_triple: 35.0={d35:.3f}  50.0={d50:.3f}')
print('asserted:', store.query_triples(head='overdraft')[:2])
print('tension(overdraft, udaap) =', round(store.tension_energy('overdraft','udaap'), 3))

## The decision is made by a CALIBRATED gate
A distance is not a decision. To call a fee value *grounded* or *fabricated* we need a boundary, and that boundary is the per-measure **geodesic** operating point that `GMSJudge.calibrate()` fits from the store's own facts and persists in `calibration.json` --- **not** a hand-picked value. A claim is grounded when its `score_triple` distance falls inside that calibrated band. (The retrieval gate in `rag_gate_calibration.json` is a different, more permissive gate; we meet it in the capstone.)

In [ ]:
import json
from knowlytix.harness.testing.judge import GMSJudge
# Per-measure operating points are fit by GMSJudge.calibrate() and persisted
# in calibration.json. We read the persisted geodesic plausibility band ---
# a calibrated distance with a known false-accept rate, never hand-picked.
tau_d = json.loads((STORE / GMSJudge.CALIBRATION_JSON).read_text())['thresholds']['geodesic']
def decide(h, r, t):
    d = store.score_triple(h, r, t)
    return round(d, 3), ('grounded' if d <= tau_d else 'fabricated')
print('calibrated geodesic threshold (distance) =', round(tau_d, 3))
print('overdraft 35.0 ->', decide('overdraft','has_fee_amount','35.0'))
print('overdraft 50.0 ->', decide('overdraft','has_fee_amount','50.0'))

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, pathlib
FIGDIR = pathlib.Path("..") / "book" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)
vals = [25, 30, 35, 45, 50, 60]        # tail values that are store entities
ds = [store.score_triple('overdraft', 'has_fee_amount', f'{v}.0') for v in vals]
fig, ax = plt.subplots(figsize=(4.6, 3))
ax.plot(vals, ds, 'o-', color='#36c')
ax.axhline(tau_d, ls='--', c='#c33', label=f'calibrated grounded band (d<={tau_d:.2f})')
ax.axvline(35, ls=':', c='#2a8', label='committed (35)')
ax.set_xlabel('tail value'); ax.set_ylabel('distance (score_triple)')
ax.set_title('Plausibility as distance; the band is calibrated')
ax.legend()
fig.tight_layout(); fig.savefig(FIGDIR / 'fig_score_distance.png', dpi=150)
from IPython.display import Image, display
display(Image(filename=str(FIGDIR / 'fig_score_distance.png')))

## Fine-tuning the two channels (real API)
The v-space (semantic) is tuned by `finetune_embedding` in **rotation** mode; the u-space (logical) by `finetune_contradiction` in **full** mode (required --- a rotation preserves angles and cannot move tension). Both insert into the store's dual embedding via `EmbeddingConfig`.

In [ ]:
import json, tempfile
from knowlytix.embedding import (EmbeddingSFTConfig, finetune_embedding,
                                 finetune_contradiction)
from knowlytix.core.config import EmbeddingConfig
from knowlytix.core.graph.embeddings import DualEmbedding
from knowlytix.core.graph.encoders import init_dual_embeddings
rows = [{'text': t, 'label': l} for t, l in [
    ('overdraft fee', 'overdraft'), ('nsf charge', 'overdraft'), ('od charge', 'overdraft'),
    ('dispute a charge', 'disputes'), ('chargeback', 'disputes'), ('unauthorized charge', 'disputes')]]
lab = Path(tempfile.mktemp(suffix='.jsonl')); lab.write_text('\n'.join(json.dumps(r) for r in rows))
names = ['overdraft', 'disputes']
v_ft = finetune_embedding(str(lab), EmbeddingSFTConfig(mode='rotation', rank=4, epochs=20, out_dim=64, device='cpu'),
                          text_col='text', label_col='label')
groups = {'has_fee_amount': ['overdraft fee', 'nsf charge'],
          'has_interest_rate': ['apr', 'interest rate'],
          'has_window_days': ['filing window', 'days to file']}
u_ft = finetune_contradiction(groups,
        EmbeddingSFTConfig(mode='full', objective='contradiction', rank=4, epochs=20, out_dim=64, device='cpu'))
pv = Path(tempfile.mktemp(suffix='.pt')); pu = Path(tempfile.mktemp(suffix='.pt'))
torch.save(v_ft.export_vectors(names), pv); torch.save(u_ft.export_vectors(names), pu)
de = DualEmbedding(num_entities=len(names), d_v=64, d_u=64, m=64)
init_dual_embeddings(de, names, EmbeddingConfig(v_vectors_path=str(pv), u_vectors_path=str(pu),
                                                warm_start=False, freeze_base=True))
row0_ok = torch.allclose(de.v_embed.weight.data[0], v_ft.export_vectors(names)['overdraft'], atol=1e-4)
print('v mode:', v_ft.adapter.mode, '| u mode:', u_ft.adapter.mode,
      '| inserted into GMS:', bool(row0_ok), '| v frozen:', not de.v_embed.weight.requires_grad)

## Before vs after fine-tuning (PCA)
The u-space contradiction objective separates logically-distinct attributes the raw encoder confuses. We embed phrasings of three attributes before (raw MiniLM) and after (contradiction-tuned), reduce each to 2-D by PCA and plot. (A v-space rotation preserves all distances, so it produces no PCA-visible separation change; its purpose is coordinate alignment --- turning the embedding into the orthonormal frame the GMS primitives are computed in --- not separation. The contradiction channel is where the geometry visibly moves.)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, pathlib
FIGDIR = pathlib.Path("..") / "book" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)
import numpy as np
from knowlytix.core.graph.encoders import encode_texts
ph = {'fee': ['overdraft fee', 'nsf charge', 'service fee'],
      'interest': ['apr', 'interest rate', 'finance charge rate'],
      'window': ['filing window', 'days to file', 'deadline in days']}
texts = [w for g in ph for w in ph[g]]; labs = [g for g in ph for _ in ph[g]]
before = encode_texts(texts, 'sentence-transformers/all-MiniLM-L6-v2', 'cpu').numpy()
after = u_ft.encode(texts).numpy()
def pca2(X):
    Xc = X - X.mean(0); _, _, Vt = np.linalg.svd(Xc, full_matrices=False); return Xc @ Vt[:2].T
cmap = {'fee': '#36c', 'interest': '#c33', 'window': '#2a8'}
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4))
for ax, (P, title) in zip(axes, [(pca2(before), 'before (raw encoder)'),
                                 (pca2(after), 'after (u-space contradiction SFT)')]):
    for g in ph:
        idx = [i for i, l in enumerate(labs) if l == g]
        ax.scatter(P[idx, 0], P[idx, 1], c=cmap[g], label=g, s=30)
    ax.set_title(title); ax.legend(fontsize=7)
fig.suptitle('Contradiction fine-tuning separates confusable attributes')
fig.tight_layout(); fig.savefig(FIGDIR / 'fig_embed_pca.png', dpi=150)
from IPython.display import Image, display
display(Image(filename=str(FIGDIR / 'fig_embed_pca.png')))

## Self-check

In [ ]:
assert d35 < d50                                  # plausibility is graded
assert decide('overdraft','has_fee_amount','35.0')[1] == 'grounded'    # calibrated
assert decide('overdraft','has_fee_amount','50.0')[1] == 'fabricated'  # calibrated
assert v_ft.adapter.mode == 'rotation' and u_ft.adapter.mode == 'full'
assert row0_ok                                   # encoders inserted into GMS
print('OK - primitives, calibration, embedding SFT')